# Passive compliance — piano playing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.path.join('../..'))
plt.style.use(os.path.join('../..', 'plot_config.mplstyle'))

OUTPUT_DIR = os.path.join('outputs', 'piano_playing_hand')

# Pull constants from the run script (single source of truth)
_ns = {}
with open('piano_playing_hand.py') as _f:
    for _line in _f:
        if _line.startswith(('K_SWEEP', 'K_STIFF', 'K_SOFT', 'PRESS_FREQUENCY')):
            exec(_line, _ns)
K_SWEEP         = [float(k) for k in _ns['K_SWEEP']]
K_STIFF         = float(_ns['K_STIFF'])
K_SOFT          = float(_ns['K_SOFT'])
PRESS_FREQUENCY = float(_ns['PRESS_FREQUENCY'])

# Software motor indices (motor_config.py)
IDX = slice(7, 9)   # index  MCP, PIP
RNG = slice(11, 13) # ring   MCP, PIP

N_GRID = 200  # interpolation points per cycle

def load_cycles(condition, k_label):
    folder = os.path.join(OUTPUT_DIR, condition, f'K_{k_label}')
    if not os.path.isdir(folder):
        return []
    files = sorted(f for f in os.listdir(folder) if f.endswith('.csv'))
    return [pd.read_csv(os.path.join(folder, f)) for f in files]

def tau_norm(df, sl):
    cols = [f'tau_{i}' for i in range(sl.start, sl.stop)]
    return np.linalg.norm(df[cols].to_numpy(), axis=1)

def interp_cycle(arr, n=N_GRID):
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, n)
    return np.interp(x_new, x_old, arr)


## Uniform condition — torque norm vs stiffness

In [ ]:
# ── Plot 1: uniform condition — torque norm vs stiffness ─────────────────────
# Each K: mean ± std of index torque norm across cycles, time-normalized to [0,1]s.

fig, ax = plt.subplots()
T = np.linspace(0, 1.0 / PRESS_FREQUENCY, N_GRID)

colors = ['#56B4E9', '#E69F00', '#000000']   # light → dark for low → high K
for color, K in zip(colors, K_SWEEP):
    label = f'K_tip = {K:.0f} N/m'
    cycles = load_cycles('uniform', f'{K:.0f}')
    if not cycles:
        continue
    traces = np.stack([interp_cycle(tau_norm(df, IDX)) for df in cycles])
    mu, sigma = traces.mean(0), traces.std(0)
    ax.plot(T, mu, color=color, label=label)
    ax.fill_between(T, mu - sigma, mu + sigma, color=color, alpha=0.15)

ax.set_xlabel('Time within cycle [s]')
ax.set_ylabel('Index torque norm [N·m]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'uniform_torque_vs_stiffness.pdf'), bbox_inches='tight')
plt.show()


## Heterogeneous condition — stiff index vs soft ring

In [ ]:
# ── Plot 2: heterogeneous condition — stiff index vs soft ring ───────────────
# Single condition (K_index = 100, K_ring = 10 N/m): both finger torques,
# mean ± std across cycles. Demonstrates asymmetric contact dynamics.

fig, ax = plt.subplots()
T = np.linspace(0, 1.0 / PRESS_FREQUENCY, N_GRID)

k_lbl = f'stiff{K_STIFF:.0f}_soft{K_SOFT:.0f}'
cycles = load_cycles('heterogeneous', k_lbl)

for color, sl, label in [
    ('#0072B2', IDX, f'Index  (K = {K_STIFF:.0f} N/m)'),
    ('#D55E00', RNG, f'Ring   (K = {K_SOFT:.0f} N/m)'),
]:
    if not cycles:
        continue
    traces = np.stack([interp_cycle(tau_norm(df, sl)) for df in cycles])
    mu, sigma = traces.mean(0), traces.std(0)
    ax.plot(T, mu, color=color, label=label)
    ax.fill_between(T, mu - sigma, mu + sigma, color=color, alpha=0.15)

ax.set_xlabel('Time within cycle [s]')
ax.set_ylabel('Finger torque norm [N·m]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'heterogeneous_index_vs_ring.pdf'), bbox_inches='tight')
plt.show()
